# ETL Pipeline - Homework Lecture 13

This notebook builds a reusable ETL pipeline for intentionally imperfect CSV data.

Flow:
1. Extract raw CSV files.
2. Explore data quality issues.
3. Transform data with deterministic cleaning rules.
4. Create analytical tables.
5. Load clean and analytical tables to SQLite and PostgreSQL.

Stack: Python, pandas, SQLite, PostgreSQL, Docker Compose, pytest.


## 0. Imports and Settings

Paths are relative, so the notebook works both locally and inside Docker.
Docker Compose overrides these paths with environment variables.


In [2]:
import os
import re
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine


ALLOWED_STATUSES = {"completed", "cancelled", "pending", "returned"}


def resolve_path(env_name: str, default: str) -> Path:
    return Path(os.getenv(env_name, default)).expanduser().resolve()


RAW_DIR = resolve_path("RAW_DIR", "data/raw")
DB_PATH = resolve_path("DB_PATH", "data/pipeline.db")
POSTGRES_DSN = os.getenv("POSTGRES_DSN")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Paths configured.")
print(f"  raw -> {RAW_DIR}")
print(f"  sqlite -> {DB_PATH}")
print(f"  postgres -> {'enabled' if POSTGRES_DSN else 'disabled'}")


Paths configured.
  raw -> C:\camp\Camp_2026_homework-main\lesson_13_DE_etl\hometask_lecture_13\data\raw
  sqlite -> C:\camp\Camp_2026_homework-main\lesson_13_DE_etl\hometask_lecture_13\data\pipeline.db
  postgres -> disabled


## 1. Extract - Read Raw Data

The extraction step reads the four input CSV files without changing them. This keeps raw data inspection separate from cleaning.


In [3]:
def read_raw_tables(raw_dir: Path) -> dict[str, pd.DataFrame]:
    required_files = {
        "customers": "customers.csv",
        "orders": "orders.csv",
        "order_items": "order_items.csv",
        "products": "products.csv",
    }
    missing = [file_name for file_name in required_files.values() if not (raw_dir / file_name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing raw files in {raw_dir}: {', '.join(missing)}")
    return {table_name: pd.read_csv(raw_dir / file_name) for table_name, file_name in required_files.items()}


In [4]:
raw_tables = read_raw_tables(RAW_DIR)

for table_name, df in raw_tables.items():
    print(f"{table_name}: {df.shape[0]} rows, {df.shape[1]} columns")
    print(df.dtypes.to_string())
    print()


customers: 357 rows, 4 columns
customer_id    float64
email              str
country            str
created_at         str

orders: 456 rows, 4 columns
order_id          int64
customer_id     float64
order_status        str
created_at          str

order_items: 914 rows, 4 columns
order_item_id    int64
order_id         int64
product_id       int64
quantity         int64

products: 155 rows, 4 columns
product_id      int64
name              str
category          str
price         float64



## 2. Explore - Identify Data Issues

This section checks the intentional issues described in the homework: duplicate keys, missing values, invalid emails, bad timestamps, invalid prices, invalid quantities, mixed statuses, and broken references.


In [5]:
def is_valid_email(value) -> bool:
    if pd.isna(value):
        return False
    return bool(re.match(r"^[^@\s]+@[^@\s]+\.[^@\s]+$", str(value).strip()))


def print_table_quality(name: str, df: pd.DataFrame, pk_col: str) -> None:
    print(f"\n{name.upper()}")
    print(f"  Rows: {len(df)}")
    print(f"  Duplicate rows: {df.duplicated().sum()}")
    print(f"  Duplicate PK ({pk_col}): {df[pk_col].duplicated().sum()}")
    nulls = df.isnull().sum()
    if nulls.any():
        print("  Missing values:")
        print(nulls[nulls > 0].to_string())


def explore_data(tables: dict[str, pd.DataFrame]) -> None:
    for name, pk_col in [("customers", "customer_id"), ("orders", "order_id"), ("order_items", "order_item_id"), ("products", "product_id")]:
        print_table_quality(name, tables[name], pk_col)

    customers_raw = tables["customers"]
    orders_raw = tables["orders"]
    items_raw = tables["order_items"]
    products_raw = tables["products"]

    bad_email_mask = ~customers_raw["email"].apply(is_valid_email)
    print(f"\nInvalid or missing customer emails: {bad_email_mask.sum()}")

    for df, col, name in [(customers_raw, "created_at", "customers"), (orders_raw, "created_at", "orders")]:
        bad = pd.to_datetime(df[col], errors="coerce").isna()
        print(f"Invalid timestamps in {name}: {bad.sum()}")

    print("Order statuses:")
    print(orders_raw["order_status"].value_counts().to_string())
    print(f"Order items with quantity <= 0: {(items_raw['quantity'] <= 0).sum()}")
    print(f"Products with price <= 0: {(products_raw['price'] <= 0).sum()}")


In [6]:
explore_data(raw_tables)



CUSTOMERS
  Rows: 357
  Duplicate rows: 0
  Duplicate PK (customer_id): 3
  Missing values:
customer_id    1
email          2

ORDERS
  Rows: 456
  Duplicate rows: 0
  Duplicate PK (order_id): 1
  Missing values:
customer_id    1

ORDER_ITEMS
  Rows: 914
  Duplicate rows: 0
  Duplicate PK (order_item_id): 1

PRODUCTS
  Rows: 155
  Duplicate rows: 0
  Duplicate PK (product_id): 1
  Missing values:
name        1
category    1

Invalid or missing customer emails: 4
Invalid timestamps in customers: 1
Invalid timestamps in orders: 1
Order statuses:
order_status
completed    329
cancelled     65
pending       60
COMPLETED      1
returned       1
Order items with quantity <= 0: 1
Products with price <= 0: 2


## 3. Transform - Clean Data

### Cleaning Decisions and Rationale

In [7]:
def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    before = len(out)
    out = out.dropna(subset=["customer_id"])
    out["customer_id"] = out["customer_id"].astype(int)
    print(f"[customers] Removed rows without PK: {before - len(out)}")

    before = len(out)
    out = out.drop_duplicates(subset=["customer_id"], keep="first")
    print(f"[customers] Removed duplicate PK rows: {before - len(out)}")

    out["email"] = out["email"].apply(lambda value: str(value).strip() if is_valid_email(value) else None)
    print(f"[customers] Invalid emails set to NULL: {out['email'].isna().sum()}")

    before = len(out)
    out["created_at"] = pd.to_datetime(out["created_at"], errors="coerce")
    out = out.dropna(subset=["created_at"])
    print(f"[customers] Removed rows with invalid timestamp: {before - len(out)}")
    print(f"[customers] Final rows: {len(out)}\n")
    return out.reset_index(drop=True)


def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    before = len(out)
    out = out.dropna(subset=["product_id"])
    out["product_id"] = out["product_id"].astype(int)
    print(f"[products] Removed rows without PK: {before - len(out)}")

    before = len(out)
    out = out.drop_duplicates(subset=["product_id"], keep="first")
    print(f"[products] Removed duplicate PK rows: {before - len(out)}")

    before = len(out)
    out = out[out["price"] > 0]
    print(f"[products] Removed rows with price <= 0: {before - len(out)}")
    print(f"[products] Final rows: {len(out)}\n")
    return out.reset_index(drop=True)


def clean_orders(df: pd.DataFrame, valid_customer_ids: set[int]) -> pd.DataFrame:
    out = df.copy()
    before = len(out)
    out = out.dropna(subset=["order_id", "customer_id"])
    out[["order_id", "customer_id"]] = out[["order_id", "customer_id"]].astype(int)
    print(f"[orders] Removed rows without required IDs: {before - len(out)}")

    before = len(out)
    out = out.drop_duplicates(subset=["order_id"], keep="first")
    print(f"[orders] Removed duplicate PK rows: {before - len(out)}")

    out["order_status"] = out["order_status"].astype(str).str.lower().str.strip()
    unknown_mask = ~out["order_status"].isin(ALLOWED_STATUSES)
    print(f"[orders] Unknown statuses set to 'unknown': {unknown_mask.sum()}")
    out.loc[unknown_mask, "order_status"] = "unknown"

    before = len(out)
    out = out[out["customer_id"].isin(valid_customer_ids)]
    print(f"[orders] Removed orphan customer references: {before - len(out)}")

    before = len(out)
    out["created_at"] = pd.to_datetime(out["created_at"], errors="coerce")
    out = out.dropna(subset=["created_at"])
    print(f"[orders] Removed rows with invalid timestamp: {before - len(out)}")
    print(f"[orders] Final rows: {len(out)}\n")
    return out.reset_index(drop=True)


def clean_order_items(df: pd.DataFrame, valid_order_ids: set[int], valid_product_ids: set[int]) -> pd.DataFrame:
    out = df.copy()
    before = len(out)
    out = out.dropna(subset=["order_item_id", "order_id", "product_id"])
    out[["order_item_id", "order_id", "product_id"]] = out[["order_item_id", "order_id", "product_id"]].astype(int)
    print(f"[order_items] Removed rows without required IDs: {before - len(out)}")

    before = len(out)
    out = out.drop_duplicates(subset=["order_item_id"], keep="first")
    print(f"[order_items] Removed duplicate PK rows: {before - len(out)}")

    neg_mask = out["quantity"] < 0
    out.loc[neg_mask, "quantity"] = out.loc[neg_mask, "quantity"].abs()
    print(f"[order_items] Negative quantities converted to absolute values: {neg_mask.sum()}")

    before = len(out)
    out = out[out["quantity"] > 0]
    print(f"[order_items] Removed rows with quantity == 0: {before - len(out)}")

    before = len(out)
    out = out[out["order_id"].isin(valid_order_ids)]
    print(f"[order_items] Removed orphan order references: {before - len(out)}")

    before = len(out)
    out = out[out["product_id"].isin(valid_product_ids)]
    print(f"[order_items] Removed orphan product references: {before - len(out)}")
    print(f"[order_items] Final rows: {len(out)}\n")
    return out.reset_index(drop=True)


def transform_data(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    customers = clean_customers(tables["customers"])
    products = clean_products(tables["products"])
    orders = clean_orders(tables["orders"], valid_customer_ids=set(customers["customer_id"]))
    order_items = clean_order_items(tables["order_items"], valid_order_ids=set(orders["order_id"]), valid_product_ids=set(products["product_id"]))
    return {"customers": customers, "products": products, "orders": orders, "order_items": order_items}


In [8]:
clean_tables = transform_data(raw_tables)

for table_name, df in clean_tables.items():
    print(f"{table_name}: {len(df)} clean rows")


[customers] Removed rows without PK: 1
[customers] Removed duplicate PK rows: 3
[customers] Invalid emails set to NULL: 2
[customers] Removed rows with invalid timestamp: 1
[customers] Final rows: 352

[products] Removed rows without PK: 0
[products] Removed duplicate PK rows: 1
[products] Removed rows with price <= 0: 2
[products] Final rows: 152

[orders] Removed rows without required IDs: 1
[orders] Removed duplicate PK rows: 1
[orders] Unknown statuses set to 'unknown': 0
[orders] Removed orphan customer references: 1
[orders] Removed rows with invalid timestamp: 1
[orders] Final rows: 452

[order_items] Removed rows without required IDs: 0
[order_items] Removed duplicate PK rows: 1
[order_items] Negative quantities converted to absolute values: 1
[order_items] Removed rows with quantity == 0: 0
[order_items] Removed orphan order references: 3
[order_items] Removed orphan product references: 3
[order_items] Final rows: 907

customers: 352 clean rows
products: 152 clean rows
orders:

## 4. Analytical Tables — Preview

build_report_frames builds the same analytical views in Python so they
can be inspected here and tested with pytest.
The **actual pipeline** creates these tables directly in the database via
CREATE TABLE AS SELECT in Section 5 — the logic lives in SQL, not in Python.


In [9]:
def build_report_frames(tables: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    customers = tables["customers"]
    products = tables["products"]
    orders = tables["orders"]
    order_items = tables["order_items"]
    billable_orders = orders[orders["order_status"] != "cancelled"].copy()

    order_lines = (
        billable_orders[["order_id", "customer_id", "created_at"]]
        .merge(order_items, on="order_id", how="inner")
        .merge(products[["product_id", "price"]], on="product_id", how="inner")
    )
    order_lines["line_total"] = order_lines["quantity"] * order_lines["price"]

    customer_orders = billable_orders.groupby("customer_id")["order_id"].nunique().rename("total_orders")
    customer_spend = order_lines.groupby("customer_id")["line_total"].sum().rename("total_spent")
    customer_summary = (
        customers[["customer_id", "email", "country"]]
        .merge(customer_orders, on="customer_id", how="left")
        .merge(customer_spend, on="customer_id", how="left")
    )
    customer_summary["total_orders"] = customer_summary["total_orders"].fillna(0).astype(int)
    customer_summary["total_spent"] = customer_summary["total_spent"].fillna(0)
    customer_summary = customer_summary.sort_values("total_spent", ascending=False).reset_index(drop=True)

    product_units = order_lines.groupby("product_id")["quantity"].sum().rename("units_sold")
    product_sales = order_lines.groupby("product_id")["line_total"].sum().rename("total_revenue")
    product_revenue = (
        products[["product_id", "name", "category", "price"]]
        .merge(product_units, on="product_id", how="left")
        .merge(product_sales, on="product_id", how="left")
    )
    product_revenue["units_sold"] = product_revenue["units_sold"].fillna(0).astype(int)
    product_revenue["total_revenue"] = product_revenue["total_revenue"].fillna(0)
    product_revenue = product_revenue.sort_values("total_revenue", ascending=False).reset_index(drop=True)

    orders_by_status = orders.groupby("order_status").size().reset_index(name="order_count")
    orders_by_status["pct"] = (orders_by_status["order_count"] * 100 / len(orders)).round(1) if len(orders) else 0
    orders_by_status = orders_by_status.sort_values("order_count", ascending=False).reset_index(drop=True)

    monthly_revenue = order_lines.copy()
    monthly_revenue["month"] = pd.to_datetime(monthly_revenue["created_at"]).dt.strftime("%Y-%m")
    monthly_revenue = (
        monthly_revenue.groupby("month")
        .agg(orders_count=("order_id", "nunique"), revenue=("line_total", "sum"))
        .reset_index()
        .sort_values("month")
    )

    return {
        "report_customer_summary": customer_summary,
        "report_product_revenue": product_revenue,
        "report_orders_by_status": orders_by_status,
        "report_monthly_revenue": monthly_revenue,
    }


In [10]:
report_tables = build_report_frames(clean_tables)

for table_name, df in report_tables.items():
    print(f"{table_name}: {len(df)} rows")
    display(df.head())


report_customer_summary: 352 rows


,customer_id,email,country,total_orders,total_spent
0,44,user44@example.com,UA,3,30203.22
1,204,user204@example.com,ES,4,26098.91
2,335,user335@example.com,UA,5,22803.45
3,111,user111@example.com,US,2,20527.57
4,323,user323@example.com,ES,3,20349.01


report_product_revenue: 152 rows


,product_id,name,category,price,units_sold,total_revenue
0,1122,Toys Product 1122,Toys,1228.00,32,39296.00
1,1039,Books Product 1039,Books,1408.70,22,30991.40
2,1047,Electronics Product 1047,Electronics,1423.62,20,28472.40
3,1008,Books Product 1008,Books,1444.41,19,27443.79
4,1125,Beauty Product 1125,Beauty,1244.53,21,26135.13


report_orders_by_status: 4 rows


,order_status,order_count,pct
0,completed,326,72.1
1,cancelled,65,14.4
2,pending,60,13.3
3,returned,1,0.2


report_monthly_revenue: 4 rows


,month,orders_count,revenue
0,2024-04,94,459264.38
1,2024-05,117,435328.60
2,2024-06,96,357695.81
3,2024-07,78,322483.22


## 5. Load — Write to SQLite and PostgreSQL

Two steps per target:
1. Load the four **clean** DataFrames with to_sql (replace mode).
2. Build **analytics tables** in the database using CREATE TABLE AS SELECT —
   join and aggregation logic lives in SQL, not in Python.

SQLite and PostgreSQL differ only in the date function for monthly bucketing
(STRFTIME vs DATE_TRUNC).


In [11]:
def make_sqlite_engine(db_path: Path = DB_PATH) -> Engine:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    return create_engine(f'sqlite:///{db_path.as_posix()}')


def make_postgres_engine(dsn: str | None = POSTGRES_DSN) -> Engine | None:
    if not dsn:
        return None
    return create_engine(dsn)


def write_clean_tables(
    engine: Engine, tables: dict[str, pd.DataFrame], label: str
) -> None:
    """Load the four clean DataFrames into the database (replace any existing data)."""
    with engine.begin() as conn:
        for name, df in tables.items():
            df.to_sql(name, conn, if_exists='replace', index=False)
            print(f'  {len(df)} rows -> {name}')
    print(f'[{label}] clean tables loaded.')


# Analytics SQL
# Built directly in the DB with CREATE TABLE AS SELECT.
# Shared queries use CASE WHEN to count/sum only non-cancelled orders.
# The only dialect difference is the date bucketing for monthly revenue.

_ANALYTICS_SHARED = [
    'DROP TABLE IF EXISTS report_customer_summary',
    '''
CREATE TABLE report_customer_summary AS
SELECT
    c.customer_id,
    c.email,
    c.country,
    COUNT(DISTINCT CASE WHEN o.order_status <> 'cancelled' THEN o.order_id END) AS total_orders,
    COALESCE(SUM(CASE WHEN o.order_status <> 'cancelled' THEN oi.quantity * p.price END), 0) AS total_spent
FROM customers c
LEFT JOIN orders o  ON c.customer_id = o.customer_id
LEFT JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
GROUP BY c.customer_id, c.email, c.country
ORDER BY total_spent DESC''',

    'DROP TABLE IF EXISTS report_product_revenue',
    '''
CREATE TABLE report_product_revenue AS
SELECT
    p.product_id,
    p.name,
    p.category,
    p.price,
    COALESCE(SUM(CASE WHEN o.order_status <> 'cancelled' THEN oi.quantity END), 0) AS units_sold,
    COALESCE(SUM(CASE WHEN o.order_status <> 'cancelled' THEN oi.quantity * p.price END), 0) AS total_revenue
FROM products p
LEFT JOIN order_items oi ON p.product_id = oi.product_id
LEFT JOIN orders o ON oi.order_id = o.order_id
GROUP BY p.product_id, p.name, p.category, p.price
ORDER BY total_revenue DESC''',

    'DROP TABLE IF EXISTS report_orders_by_status',
    '''
CREATE TABLE report_orders_by_status AS
SELECT
    order_status,
    COUNT(*) AS order_count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM orders), 1) AS pct
FROM orders
GROUP BY order_status
ORDER BY order_count DESC''',
]

# Monthly revenue — differs only in the date bucketing function
_MONTHLY_POSTGRES = [
    'DROP TABLE IF EXISTS report_monthly_revenue',
    '''
CREATE TABLE report_monthly_revenue AS
SELECT
    TO_CHAR(DATE_TRUNC('month', o.created_at), 'YYYY-MM') AS month,
    COUNT(DISTINCT o.order_id) AS orders_count,
    COALESCE(SUM(CASE WHEN o.order_status <> 'cancelled' THEN oi.quantity * p.price END), 0) AS revenue
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
GROUP BY DATE_TRUNC('month', o.created_at)
ORDER BY month''',
]

_MONTHLY_SQLITE = [
    'DROP TABLE IF EXISTS report_monthly_revenue',
    '''
CREATE TABLE report_monthly_revenue AS
SELECT
    STRFTIME('%Y-%m', o.created_at) AS month,
    COUNT(DISTINCT o.order_id) AS orders_count,
    COALESCE(SUM(CASE WHEN o.order_status <> 'cancelled' THEN oi.quantity * p.price END), 0) AS revenue
FROM orders o
LEFT JOIN order_items oi ON o.order_id = oi.order_id
LEFT JOIN products p ON oi.product_id = p.product_id
GROUP BY STRFTIME('%Y-%m', o.created_at)
ORDER BY month''',
]


def build_analytics(engine: Engine, label: str) -> None:
    """Create the four analytics tables inside the database using SQL."""
    dialect = engine.dialect.name
    stmts = _ANALYTICS_SHARED + (
        _MONTHLY_POSTGRES if dialect == 'postgresql' else _MONTHLY_SQLITE
    )
    with engine.begin() as conn:
        for sql in stmts:
            conn.execute(text(sql))
    print(f'[{label}] analytics tables created.')


In [12]:
print('    SQLite    ')
sqlite_engine = make_sqlite_engine(DB_PATH)
write_clean_tables(sqlite_engine, clean_tables, 'sqlite')
build_analytics(sqlite_engine, 'sqlite')

postgres_engine = make_postgres_engine(POSTGRES_DSN)
if postgres_engine is not None:
    print('\n    PostgreSQL    ')
    write_clean_tables(postgres_engine, clean_tables, 'postgres')
    build_analytics(postgres_engine, 'postgres')
else:
    print('\nPostgreSQL skipped — POSTGRES_DSN not set.')


    SQLite    
  352 rows -> customers
  152 rows -> products
  452 rows -> orders
  907 rows -> order_items
[sqlite] clean tables loaded.
[sqlite] analytics tables created.

PostgreSQL skipped — POSTGRES_DSN not set.


## 6. Summary


In [13]:
print('Pipeline completed.')
print('\nClean tables:')
for name, df in clean_tables.items():
    print(f'  {name}: {len(df)} rows')

print('\nAnalytics tables (built in DB via SQL):')
for name in [
    'report_customer_summary',
    'report_product_revenue',
    'report_orders_by_status',
    'report_monthly_revenue',
]:
    print(f'  {name}')

print(f'\nSQLite: {DB_PATH}')
print(f'PostgreSQL: {"enabled" if POSTGRES_DSN else "disabled"}')


Pipeline completed.

Clean tables:
  customers: 352 rows
  products: 152 rows
  orders: 452 rows
  order_items: 907 rows

Analytics tables (built in DB via SQL):
  report_customer_summary
  report_product_revenue
  report_orders_by_status
  report_monthly_revenue

SQLite: C:\camp\Camp_2026_homework-main\lesson_13_DE_etl\hometask_lecture_13\data\pipeline.db
PostgreSQL: disabled
